In [1]:
import os
os.environ["NUMEXPR_MAX_THREADS"] = "24" 

In [2]:
from htmd.ui import *
from htmd.home import home
config(viewer='webgl',njobs=24)

New HTMD version (2.3.2) is available. You are currently on (2.2.0).There are several methods to update:    - Create a new conda env. using `conda create -n htmd2.3.2 htmd=2.3.2 -c acellera -c conda-forge`    - Create a brand new conda installation and run `conda install htmd -c acellera -c conda-forge`    - Run: `conda update htmd -c acellera -c conda-forge` (NOT RECOMMENDED!)

Registration is not approved: <html>
<head><title>405 Not Allowed</title></head>
<body>
<center><h1>405 Not Allowed</h1></center>
<hr><center>openresty</center>
</body>
</html>

Academic License 
HTMD Software Academic License Agreement v1.1

The HTMD software ("Software") has been developed by the contributing
researchers from Acellera and made available through Acellera for your
internal, non-profit research use.

Acellera allows researchers at your institution to run, display, copy and
modify Software on the following conditions:

1. The Software remains at your institution and is not published, distributed,

In [3]:
pwd

'/home/saleh/Research_2/NON_EQ/SOAT1/BUILD'

## Build the protein

In [9]:
streams= ['toppar_all36_lipid_cholesterol_model_1.str','toppar_all36_lipid_detergent.str']
topos  = charmm.defaultTopo() 
params = charmm.defaultParam()
prot = Molecule("model_final_soat1.pdb")
prot.remove('resname ROV COA 3VV OLA')
prot.remove('chain C and resid 702 703')
prot.remove('chain D and resid 704')
prot = systemPrepare(prot)
prot.set('resname','CHL1','resname CLR')
prot.rotateBy(rotationMatrix([0, 0, 1],-0.45))
pcenter = np.mean(prot.get('coords','protein'), axis=0)
prot= autoSegment(prot)
prot = charmm.build(prot, topo=topos, param=params, stream= streams, outdir= './prot' ,ionize=False)

2023-09-01 15:04:07,388 - moleculekit.molecule - INFO - Removed 208 atoms. 7410 atoms remaining in the molecule.
2023-09-01 15:04:07,421 - moleculekit.molecule - INFO - Removed 56 atoms. 7354 atoms remaining in the molecule.
2023-09-01 15:04:07,453 - moleculekit.molecule - INFO - Removed 28 atoms. 7326 atoms remaining in the molecule.
2023-09-01 15:04:07,498 - moleculekit.tools.preparation - WARNING - Both chains and segments are defined in Molecule.chain / Molecule.segid, however they are inconsistent. Protein preparation will use the chain information.



---- Molecule chain report ----
Chain C:
    First residue: GLN   118  
    Final residue: CLR   704  
Chain D:
    First residue: GLN   118  
    Final residue: CLR   703  
---- End of chain report ----



2023-09-01 15:04:12,642 - moleculekit.tools.preparation - WARNING - The following residues have not been optimized: CLR
2023-09-01 15:04:13,185 - moleculekit.tools.preparation - INFO - Modified residue HIS   137 C to HID
2023-09-01 15:04:13,186 - moleculekit.tools.preparation - INFO - Modified residue HIS   143 C to HID
2023-09-01 15:04:13,186 - moleculekit.tools.preparation - INFO - Modified residue HIS   205 C to HID
2023-09-01 15:04:13,186 - moleculekit.tools.preparation - INFO - Modified residue HIS   215 C to HID
2023-09-01 15:04:13,187 - moleculekit.tools.preparation - INFO - Modified residue HIS   223 C to HIE
2023-09-01 15:04:13,187 - moleculekit.tools.preparation - INFO - Modified residue LYS   266 C to LYN
2023-09-01 15:04:13,187 - moleculekit.tools.preparation - INFO - Modified residue HIS   268 C to HID
2023-09-01 15:04:13,187 - moleculekit.tools.preparation - INFO - Modified residue CYS   365 C to CYM
2023-09-01 15:04:13,188 - moleculekit.tools.preparation - INFO - Modifie

Disulfide Bond between: UniqueResidueID<resname: 'CYX', chain: 'C', resid: 528, insertion: '', segid: 'P0'>
                   and: UniqueResidueID<resname: 'CYX', chain: 'C', resid: 546, insertion: '', segid: 'P0'>

Disulfide Bond between: UniqueResidueID<resname: 'CYX', chain: 'D', resid: 528, insertion: '', segid: 'P2'>
                   and: UniqueResidueID<resname: 'CYX', chain: 'D', resid: 546, insertion: '', segid: 'P2'>



2023-09-01 15:04:14,136 - htmd.builder.charmm - INFO - Starting the build.
2023-09-01 15:04:14,253 - htmd.builder.charmm - WARNING - Failed to set coordinates for 24 atoms.
2023-09-01 15:04:14,253 - htmd.builder.charmm - WARNING - Failed to guess coordinates for 2 atoms due to bad angles.
2023-09-01 15:04:14,253 - htmd.builder.charmm - WARNING - Poorly guessed coordinates for 28 atoms.
2023-09-01 15:04:14,253 - htmd.builder.charmm - WARNING - Please check /home/saleh/Research_2/NON_EQ/SOAT1/BUILD/prot/log.txt for further information.
2023-09-01 15:04:14,254 - htmd.builder.charmm - INFO - Finished building.


In [10]:
#prot
pcenter

array([155.53214 ,  38.626213, 144.90984 ], dtype=float32)

In [11]:
#Sanity check
prot.view()

NGLWidget()

## Embed the protein into a membrane

Center the membrane onto the protein center and embed the protein into the membrane. Then add water and build the system

In [12]:
#Use a POPC+CHL1 membrane built with htmd
memb = Molecule('memb_popc_chl1_170.pdb')
mcenter = np.mean(memb.get('coords'),axis=0)
memb.moveBy(pcenter-mcenter)
testcenter = np.mean(memb.get('coords'),axis=0)
mol = prot.copy()

2023-09-01 15:04:57,688 - moleculekit.readers - WARNING - Non-integer values were read from the PDB "serial" field. Dropping PDB values and assigning new ones.
2023-09-01 15:04:57,728 - moleculekit.readers - WARNING - Reading PDB file with more than 99999 atoms. Bond information can be wrong.


In [13]:
print(mcenter)
print(testcenter)

[2.1540503e-05 4.6414745e-05 3.2448184e-04]
[155.5334   38.62586 144.9103 ]


In [14]:
mol.append(memb, collisions=True)  # Append membrane and remove colliding atoms
#mol = embed(mol,memb)

2023-09-01 15:05:37,237 - moleculekit.molecule - INFO - Removed 241 residues from appended Molecule due to collisions.


In [ ]:
#Sanity check
mol.write('test_embedd.pdb')

In [ ]:
#Sanity check
mol.view(sel='resname HOH')

In [15]:
#Add water
coo = mol.get('coords','name P')
m = np.min(coo,axis=0) + [2,2,-20]
M = np.max(coo,axis=0) + [-2,-2,20]
minmax=np.vstack((m,M))

In [16]:
print(coo)
print(m)
print(M)
print(minmax)

[[ 97.178116   39.247166  165.17552  ]
 [212.94612    -4.7038345 163.17952  ]
 [204.20412     6.1221657 165.57852  ]
 ...
 [126.49812   -18.474834  124.589516 ]
 [110.361115  112.43017   124.32951  ]
 [228.32611    47.751167  126.23351  ]]
[ 78.51111603 -41.23383331 100.30851746]
[233.65612793 117.85916901 188.58052063]
[[ 78.51111603 -41.23383331 100.30851746]
 [233.65612793 117.85916901 188.58052063]]


In [17]:
mol = solvate(mol, minmax=np.vstack((m,M)))

2023-09-01 15:06:26,101 - htmd.builder.solvate - INFO - Using water pdb file at: /home/saleh/miniconda3/envs/htmd/lib/python3.10/site-packages/htmd/share/solvate/wat.pdb
2023-09-01 15:06:27,108 - htmd.builder.solvate - INFO - Replicating 18 water segments, 3 by 3 by 2
Solvating: 100%|████████████████████████████████| 18/18 [02:39<00:00,  8.86s/it]
2023-09-01 15:09:11,204 - htmd.builder.solvate - INFO - 32769 water molecules were added to the system.


In [18]:
#Build
mol = charmm.build(mol, topo=topos, param=params,stream= streams, outdir='./build', saltconc=0.15)

2023-09-01 15:09:14,129 - htmd.builder.charmm - INFO - Writing out segments.
2023-09-01 15:09:18,565 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2023-09-01 15:09:18,999 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2023-09-01 15:09:19,325 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2023-09-01 15:09:19,659 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2023-09-01 15:09:19,985 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2023-09-01 15:09:20,318 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2023-09-01 15:09:20,654 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will b

Disulfide Bond between: UniqueResidueID<resname: 'CYS', chain: 'C', resid: 528, insertion: '', segid: 'P0'>
                   and: UniqueResidueID<resname: 'CYS', chain: 'C', resid: 546, insertion: '', segid: 'P0'>

Disulfide Bond between: UniqueResidueID<resname: 'CYS', chain: 'D', resid: 528, insertion: '', segid: 'P2'>
                   and: UniqueResidueID<resname: 'CYS', chain: 'D', resid: 546, insertion: '', segid: 'P2'>



2023-09-01 15:09:31,875 - htmd.builder.charmm - INFO - Starting the build.
2023-09-01 15:09:32,854 - htmd.builder.charmm - INFO - Finished building.
2023-09-01 15:09:34,650 - moleculekit.readers - WARNING - Non-integer values were read from the PDB "serial" field. Dropping PDB values and assigning new ones.
2023-09-01 15:09:34,719 - moleculekit.readers - WARNING - Reading PDB file with more than 99999 atoms. Bond information can be wrong.
2023-09-01 15:09:39,427 - htmd.builder.ionize - INFO - Adding 30 anions + 0 cations for neutralizing and 186 ions for the given salt concentration 0.15 M.
2023-09-01 15:11:04,046 - htmd.builder.charmm - INFO - Writing out segments.
2023-09-01 15:11:08,581 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2023-09-01 15:11:09,023 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.
2023-09-01 15:11:09,353 - moleculekit.writers - WARNI

In [14]:
#Sanity check
mol.write('test_solvate_P.pdb')

2022-12-16 17:39:14,825 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.


In [19]:
#Sanity check
mol.view(sel='protein ')

2023-09-01 15:12:47,696 - moleculekit.writers - WARNING - Field "serial" of PDB overflows. Your data will be truncated to 5 characters.


NGLWidget()

In [20]:
from htmd.protocols.equilibration_v3 import Equilibration


In [21]:
md = Equilibration()
md.runtime = 40
md.timeunits = 'ns'
md.temperature = 310
md.useconstantratio = True  # only for membrane sims
# # Add a 10A flat bottom potential to prevent the ligand from diffusing from original position during equilibration
# width = np.array([10, 10, 10])
# flatbot = GroupRestraint('segname L and noh', width, [(5, '0ns')])
#md.restraints = md.defaultEquilRestraints('1ns')
md.write('./build/', './equil')


2023-09-01 15:12:55,716 - moleculekit.readers - WARNING - Non-integer values were read from the PDB "serial" field. Dropping PDB values and assigning new ones.
2023-09-01 15:12:55,786 - moleculekit.readers - WARNING - Reading PDB file with more than 99999 atoms. Bond information can be wrong.


In [22]:
%ls equil/

input  parameters  run.sh*  structure.pdb  structure.psf


## Edit the input file to increase the minimization to 5000 steps

In [19]:
local = LocalGPUQueue()
local.submit('./equil/')
local.wait()

2022-12-16 17:52:07,672 - jobqueues.util - INFO - Trying to determine all GPU devices
2022-12-16 17:52:07,693 - jobqueues.localqueue - INFO - Using GPU devices 0
2022-12-16 17:52:07,694 - jobqueues.util - INFO - Trying to determine all GPU devices
2022-12-16 17:52:07,711 - jobqueues.localqueue - INFO - Queueing /home/saleh/Research_2/SOAT_CPX_POPC_CHL/SOAT1/BUILD/equil
2022-12-16 17:52:07,712 - jobqueues.localqueue - INFO - Running /home/saleh/Research_2/SOAT_CPX_POPC_CHL/SOAT1/BUILD/equil on device 0
2022-12-17 06:00:40,710 - jobqueues.localqueue - INFO - Completed /home/saleh/Research_2/SOAT_CPX_POPC_CHL/SOAT1/BUILD/equil


In [4]:
from htmd.protocols.production_v6 import Production

In [5]:
md = Production()
md.runtime = 50
md.timeunits = 'ns'
md.temperature  = 310
md.acemd.bincoordinates = 'output.coor'
md.acemd.extendedsystem  = 'output.xsc'
md.write('./equil/','./prod')

2023-09-05 12:05:51,709 - moleculekit.readers - WARNING - Non-integer values were read from the PDB "serial" field. Dropping PDB values and assigning new ones.
2023-09-05 12:05:51,779 - moleculekit.readers - WARNING - Reading PDB file with more than 99999 atoms. Bond information can be wrong.
2023-09-05 12:06:47,225 - htmd.protocols.production_v6 - WARNING - Lipids detected in input structure. We highly recommend setting useconstantratio=True for membrane simulations.


In [6]:
%ls prod/


input  input.coor  input.xsc  parameters  run.sh*  structure.pdb  structure.psf


In [ ]:
local = LocalGPUQueue()
local.submit('./prod/')
local.wait()

In [ ]:
%ls prod/output.xtc